# Mandalay CPCB AQI — XGBoost Delta24 (2026)

**Run cells from top to bottom.** Only the seven pollutant/timestamp columns are required; the script calculates CPCB AQI itself, following the existing Mandalay dashboard `cpcb-2014-rolling-v1` calculation.

Input: `timestamp, pm25, pm10, co_ug_m3, no2_ug_m3, so2_ug_m3, o3_ug_m3`

Timestamps must be **Asia/Yangon** local time, concentrations **µg/m³**, and every timestamp must be a whole hour. Missing hourly slots are preserved. No US AQI is used.

**Note:** CPCB estimates from CAMS modeled concentrations are not official station AQI. Missing/sparse observations can reduce usable training examples.

In [ ]:
%pip -q install "xgboost>=2.0" "scikit-learn>=1.3" pandas numpy openpyxl joblib requests

In [1]:
from google.colab import drive
drive.mount("/content/drive")

Mounted at /content/drive


## 1. Set the input path

Edit only `INPUT_FILE` if your workbook has a different name. The file must contain **exactly these seven columns**. The output folder is deliberately separate from your old US-AQI models.

In [2]:
from pathlib import Path
import os

INPUT_FILE = Path("/content/drive/MyDrive/AQI-Test/datasets/mandalay_aqi_2026_hourly.xlsx")
OUTPUT_DIR = Path("/content/drive/MyDrive/AQI-Test/XGB_Mandalay_CPCB_Delta24_2026")

assert INPUT_FILE.is_file(), f"Input file missing: {INPUT_FILE} — change INPUT_FILE above to match your actual file."
os.environ["MANDALAY_INPUT"] = str(INPUT_FILE)
os.environ["MANDALAY_OUTPUT"] = str(OUTPUT_DIR)
print("Input:", INPUT_FILE)
print("Output:", OUTPUT_DIR)

Input: /content/drive/MyDrive/AQI-Test/datasets/mandalay_aqi_2026_hourly.xlsx
Output: /content/drive/MyDrive/AQI-Test/XGB_Mandalay_CPCB_Delta24_2026


## 2. Fetch the exact CPCB calculation used by the Mandalay dashboard

This downloads `cpcb.py` from your own GitHub repository. Verify that it prints `cpcb-2014-rolling-v1`. Internet access is needed for this cell.

In [3]:
import requests, sys
from pathlib import Path
url = "https://raw.githubusercontent.com/joej011218-cpu/Mandalay-AQI-Dashboard/main/cpcb.py"
r = requests.get(url, timeout=30)
r.raise_for_status()
Path("/content/cpcb.py").write_text(r.text, encoding="utf-8")
sys.path.insert(0, "/content")
from cpcb import VERSION
assert VERSION == "cpcb-2014-rolling-v1", f"Unexpected methodology version: {VERSION}"
print("Loaded CPCB calculation version:", VERSION)

Loaded CPCB calculation version: cpcb-2014-rolling-v1


## 3. Training logic

The next cell defines the complete loading, CPCB computation, 198 feature engineering, chronologically split training and model-saving logic. It **does not** require another Python file.

In [4]:
"""Mandalay CPCB Delta24 XGBoost training, input pollutant columns only.

Run in Colab. Uses the same `cpcb.py` as the deployed Mandalay dashboard.
Do not edit CPCB methodology separately here; retrain if its version changes.
"""
from pathlib import Path
from datetime import timedelta
import os, json, joblib
import numpy as np
import pandas as pd
from xgboost import XGBRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from cpcb import calculate, VERSION

INPUT_PATH = Path(os.environ.get('MANDALAY_INPUT', '/content/drive/MyDrive/AQI-Test/datasets/mandalay_pollutants_2026_hourly.xlsx'))
OUTPUT_ROOT = Path(os.environ.get('MANDALAY_OUTPUT', '/content/drive/MyDrive/AQI-Test/XGB_Mandalay_CPCB_Delta24_2026'))
EXPECTED_COLUMNS = ['timestamp','pm25','pm10','co_ug_m3','no2_ug_m3','so2_ug_m3','o3_ug_m3']
POLLUTANTS = ['pm25','pm10','co','no2','so2','o3']
AQI_LAGS = [1,2,3,6,12,24,48,72,168]
POLLUTANT_LAGS = [1,2,3,6,12,24,48,72,168]
ROLLING_WINDOWS = [3,6,12,24]
TREND_WINDOWS = [3,6,12]
CHANGE_LAGS = [1,3,6,12,24]


def load_data():
    if not INPUT_PATH.exists():
        raise FileNotFoundError(f'Dataset not found: {INPUT_PATH}. Set MANDALAY_INPUT to your actual file path.')
    df = pd.read_excel(INPUT_PATH) if INPUT_PATH.suffix.lower() in ('.xlsx','.xls') else pd.read_csv(INPUT_PATH)
    df.columns = df.columns.astype(str).str.strip().str.lower()
    if list(df.columns) != EXPECTED_COLUMNS:
        if set(df.columns) != set(EXPECTED_COLUMNS):
            raise ValueError(f'Expected exactly {EXPECTED_COLUMNS}; found {df.columns.tolist()}')
        df = df[EXPECTED_COLUMNS]
    # Naive source timestamps are explicitly interpreted as Myanmar local time.
    ts = pd.to_datetime(df['timestamp'], errors='raise')
    if ts.dt.tz is None:
        ts = ts.dt.tz_localize('Asia/Yangon')
    else:
        ts = ts.dt.tz_convert('Asia/Yangon')
    df['timestamp'] = ts
    df = df.sort_values('timestamp')
    if df['timestamp'].duplicated().any():
        raise ValueError('Duplicate timestamps found; resolve before training.')
    if ((df['timestamp'].dt.minute!=0) | (df['timestamp'].dt.second!=0)).any():
        raise ValueError('Timestamps must represent whole-hour observations.')
    for c in EXPECTED_COLUMNS[1:]:
        df[c] = pd.to_numeric(df[c], errors='coerce')
        df.loc[(df[c] < 0) | (~np.isfinite(df[c])), c] = np.nan
    df = df.set_index('timestamp').asfreq('h').reset_index()
    df['pm25'] = df['pm25'].astype(float)
    df['pm10'] = df['pm10'].astype(float)
    df['no2'] = df['no2_ug_m3'].astype(float)
    df['so2'] = df['so2_ug_m3'].astype(float)
    df['o3'] = df['o3_ug_m3'].astype(float)
    df['co'] = df['co_ug_m3'].astype(float) / 1000.0  # ML feature in mg/m3, CPCB engine receives µg/m3
    return df


def calculate_targets(df):
    """Call the actual production CPCB logic, preserving missing values."""
    from cpcb import POLLUTANTS as CPCB_KEYS
    mapping = {'pm2_5':'pm25','pm10':'pm10','carbon_monoxide':'co_ug_m3',
               'nitrogen_dioxide':'no2_ug_m3','sulphur_dioxide':'so2_ug_m3','ozone':'o3_ug_m3'}
    # UTC-aware dict keys avoid mixing different local or naive timestamp conventions.
    utc = df['timestamp'].dt.tz_convert('UTC')
    history = {t.to_pydatetime(): {key: (float(row[mapping[key]]) if pd.notna(row[mapping[key]]) else None)
                for key in CPCB_KEYS} for t,(_,row) in zip(utc,df.iterrows())}
    values = []
    for t in utc:
        result = calculate(t.to_pydatetime(), history)
        values.append(result['cpcb_aqi'])
    df = df.copy()
    df['aqi'] = pd.Series(values,dtype='float64')
    print('CPCB calculation:', VERSION, 'eligible rows:', df['aqi'].notna().sum(), '/', len(df))
    return df


def build_features(df):
    """Historical features available at forecast origin t, never t+h."""
    data = df.copy()
    features = []
    def add(name, series):
        data[name] = series
        features.append(name)
    for lag in AQI_LAGS:
        add(f'aqi_lag_{lag}',data['aqi'].shift(lag))
    for p in POLLUTANTS:
        for lag in POLLUTANT_LAGS:
            add(f'{p}_lag_{lag}',data[p].shift(lag))
    for w in ROLLING_WINDOWS:
        roll = data['aqi'].rolling(w,min_periods=w)
        for label,series in [('mean',roll.mean()),('max',roll.max()),('min',roll.min()),('std',roll.std())]:
            add(f'aqi_roll_{w}h_{label}',series)
    for p in POLLUTANTS:
        for w in ROLLING_WINDOWS:
            roll = data[p].rolling(w,min_periods=w)
            add(f'{p}_roll_{w}h_mean',roll.mean())
            add(f'{p}_roll_{w}h_std',roll.std())
    for p in POLLUTANTS:
        for lag in CHANGE_LAGS:
            add(f'{p}_change_{lag}h',data[p]-data[p].shift(lag))
    for lag in CHANGE_LAGS:
        add(f'aqi_change_{lag}h',data['aqi']-data['aqi'].shift(lag))
    for p in POLLUTANTS:
        for w in TREND_WINDOWS:
            add(f'{p}_trend_{w}h',data[p].diff(w)/w)
    for w in TREND_WINDOWS:
        add(f'aqi_trend_{w}h',data['aqi'].diff(w)/w)
    for p in POLLUTANTS:
        add(f'{p}_acceleration',data[p].diff(1)-data[p].diff(3)/3)
    add('aqi_acceleration',data['aqi'].diff(1)-data['aqi'].diff(3)/3)
    add('pm25_pm10_ratio',data['pm25']/data['pm10'].replace(0,np.nan))
    add('pm25_dominance',data['pm25']/(data['pm25']+data['pm10']+1e-6))
    t=data['timestamp'].dt
    for name,x,period in [('hour',t.hour,24),('day',t.dayofweek,7),('year',t.dayofyear,365.25)]:
        add(f'sin_{name}',np.sin(2*np.pi*x/period))
        add(f'cos_{name}',np.cos(2*np.pi*x/period))
    data[features]=data[features].replace([np.inf,-np.inf],np.nan)
    print('Feature count:',len(features))
    return data,features


def train():
    df=calculate_targets(load_data())
    data,cols=build_features(df)
    OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)
    # Chronological temporal boundaries fixed for all horizons (train 70%, validation 15%, test 15%).
    n=len(data)
    train_end=data['timestamp'].iloc[int(n*.70)]
    val_end=data['timestamp'].iloc[int(n*.85)]
    rows=[]
    for horizon in range(1,25):
        work=data.copy()
        work['target_timestamp']=work['timestamp']+pd.Timedelta(hours=horizon)
        work['future_aqi']=work['aqi'].shift(-horizon)
        work['delta']=work['future_aqi']-work['aqi']
        valid=work.dropna(subset=cols+['aqi','future_aqi','delta']).copy()
        train_mask=(valid['target_timestamp']<train_end)
        val_mask=(valid['timestamp']>=train_end)&(valid['target_timestamp']<val_end)
        test_mask=(valid['timestamp']>=val_end)
        train_df=valid.loc[train_mask]
        val_df=valid.loc[val_mask]
        test_df=valid.loc[test_mask]
        if min(len(train_df),len(val_df),len(test_df))<30:
            raise ValueError(f'Horizon {horizon}: insufficient usable samples in one split')
        model=XGBRegressor(n_estimators=300,learning_rate=.05,max_depth=5,min_child_weight=1,
            subsample=.85,colsample_bytree=.85,reg_alpha=0.,reg_lambda=1.,
            objective='reg:squarederror',tree_method='hist',random_state=42+horizon,n_jobs=2)
        model.fit(train_df[cols].astype('float32'),train_df['delta'])
        metrics={'horizon':horizon,'train_n':len(train_df),'validation_n':len(val_df),'test_n':len(test_df)}
        for split_name,sample in [('validation',val_df),('test',test_df)]:
            delta_pred=model.predict(sample[cols].astype('float32'))
            pred=np.clip(sample['aqi'].to_numpy()+delta_pred,0,500)
            actual=sample['future_aqi'].to_numpy()
            metrics[f'{split_name}_mae']=float(mean_absolute_error(actual,pred))
            metrics[f'{split_name}_rmse']=float(np.sqrt(mean_squared_error(actual,pred)))
            metrics[f'{split_name}_r2']=float(r2_score(actual,pred))
            metrics[f'{split_name}_persistence_mae']=float(mean_absolute_error(actual,sample['aqi']))
        folder=OUTPUT_ROOT/f'horizon_{horizon:02d}h'
        folder.mkdir(parents=True,exist_ok=True)
        joblib.dump(model,folder/'xgboost_delta_model.joblib')
        joblib.dump(cols,folder/'feature_columns.joblib')
        config={'model':'XGBoost_DeltaOnly','city':'Mandalay, Myanmar','aqi_type':'Indian CPCB AQI',
                'calculation_version':VERSION,'forecast_horizon':horizon,
                'target':'Future CPCB AQI - Current CPCB AQI',
                'final_prediction':'Current CPCB AQI + Predicted Delta',
                'co_training_unit':'mg/m3','other_pollutant_unit':'ug/m3',
                'expected_feature_count':len(cols),'timestamp_timezone':'Asia/Yangon',
                'features_include_current_hour':True,'training_data_columns':EXPECTED_COLUMNS,
                'trained_until_exclusive':str(train_end),'validation_until_exclusive':str(val_end),
                'metrics':metrics}
        joblib.dump(config,folder/'model_config.joblib')
        rows.append(metrics)
        print(f"h+{horizon:02d}: test RMSE={metrics['test_rmse']:.3f}, MAE={metrics['test_mae']:.3f}, R2={metrics['test_r2']:.3f}",flush=True)
    pd.DataFrame(rows).to_csv(OUTPUT_ROOT/'evaluation_24h.csv',index=False)
    (OUTPUT_ROOT/'training_manifest.json').write_text(json.dumps({
      'city':'Mandalay, Myanmar','aqi_standard':'Indian CPCB AQI',
      'calculation_version':VERSION,'feature_columns':cols,
      'input_file':str(INPUT_PATH),'train_end':str(train_end),'validation_end':str(val_end),
      'limitations':'CAMS modeled concentrations; not official station measurements. No NH3 or lead.',
      'note':'Missing CPCB AQI is never filled or replaced by US AQI.'},indent=2))
    print('Saved models to:',OUTPUT_ROOT)



## 4. Preview the dataset (optional but recommended)

In [5]:
raw = load_data()
print("Rows:", len(raw), "Period:", raw.timestamp.min(), "→", raw.timestamp.max())
print(raw[["timestamp", "pm25", "pm10", "co", "no2", "so2", "o3"]].head(3).to_string(index=False))
print("Missing values by pollutant:")
print(raw[["pm25", "pm10", "co", "no2", "so2", "o3"]].isna().sum())

Rows: 6768 Period: 2026-01-01 00:00:00+06:30 → 2026-10-09 23:00:00+06:30
                timestamp  pm25  pm10    co  no2  so2   o3
2026-01-01 00:00:00+06:30  25.3  25.9 0.677 13.5  7.4 54.0
2026-01-01 01:00:00+06:30  23.4  24.0 0.632 14.0  6.4 48.0
2026-01-01 02:00:00+06:30  22.4  23.0 0.606 14.8  5.7 41.0
Missing values by pollutant:
pm25    0
pm10    0
co      0
no2     0
so2     0
o3      0
dtype: int64


## 5. Train +1 through +24 hours

The code saves 24 sets of `.joblib` files, a training manifest, and test metrics. **Run this cell once**. If it fails partway, address the error and rerun; existing files will be overwritten in this CPCB-only output folder.

In [6]:
train()

CPCB calculation: cpcb-2014-rolling-v1 eligible rows: 6753 / 6768
Feature count: 198


/tmp/ipykernel_1354/345969259.py:84: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  data[name] = series
/tmp/ipykernel_1354/345969259.py:84: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  data[name] = series
/tmp/ipykernel_1354/345969259.py:84: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  data[name] =

h+01: test RMSE=1.968, MAE=1.123, R2=0.992
h+02: test RMSE=3.425, MAE=1.983, R2=0.976
h+03: test RMSE=4.780, MAE=2.882, R2=0.953
h+04: test RMSE=5.972, MAE=3.754, R2=0.927
h+05: test RMSE=7.090, MAE=4.620, R2=0.898
h+06: test RMSE=8.086, MAE=5.506, R2=0.867
h+07: test RMSE=8.598, MAE=6.025, R2=0.850
h+08: test RMSE=9.730, MAE=7.078, R2=0.808
h+09: test RMSE=10.312, MAE=7.599, R2=0.784
h+10: test RMSE=10.646, MAE=7.949, R2=0.770
h+11: test RMSE=11.070, MAE=8.567, R2=0.751
h+12: test RMSE=11.191, MAE=8.580, R2=0.746
h+13: test RMSE=10.929, MAE=8.359, R2=0.758
h+14: test RMSE=11.338, MAE=8.694, R2=0.740
h+15: test RMSE=11.302, MAE=8.684, R2=0.741
h+16: test RMSE=11.335, MAE=8.540, R2=0.740
h+17: test RMSE=11.195, MAE=8.475, R2=0.746
h+18: test RMSE=11.382, MAE=8.679, R2=0.738
h+19: test RMSE=11.401, MAE=8.674, R2=0.737
h+20: test RMSE=11.115, MAE=8.450, R2=0.751
h+21: test RMSE=11.463, MAE=8.721, R2=0.735
h+22: test RMSE=11.617, MAE=8.899, R2=0.728
h+23: test RMSE=11.522, MAE=8.680, R2=0.

## 6. View metrics and verify saved models

In [7]:
import pandas as pd
metrics = pd.read_csv(OUTPUT_DIR / "evaluation_24h.csv")
display(metrics[["horizon","train_n","validation_n","test_n","test_mae","test_rmse","test_r2","test_persistence_mae"]])
for h in range(1,25):
    folder=OUTPUT_DIR / f"horizon_{h:02d}h"
    for name in ("xgboost_delta_model.joblib","feature_columns.joblib","model_config.joblib"):
        assert (folder / name).is_file(), f"Missing {folder/name}"
print("Verified: 24 horizon folders / 72 .joblib artifacts")

,horizon,train_n,validation_n,test_n,test_mae,test_rmse,test_r2,test_persistence_mae
0,1,4553,1014,1015,1.122767,1.967962,0.992083,4.812808
1,2,4552,1013,1014,1.983485,3.425174,0.976030,9.381657
2,3,4551,1012,1013,2.881587,4.779860,0.953352,13.611056
3,4,4550,1011,1012,3.754239,5.972455,0.927229,17.419960
4,5,4549,1010,1011,4.620498,7.090191,0.897538,20.782394
5,6,4548,1009,1010,5.506159,8.086047,0.866865,23.679208
6,7,4547,1008,1009,6.024996,8.597881,0.849623,26.099108
7,8,4546,1007,1008,7.077776,9.730162,0.807576,28.066468
8,9,4545,1006,1007,7.598714,10.312372,0.784016,29.636544
9,10,4544,1005,1006,7.949203,10.645840,0.769964,30.730616


Verified: 24 horizon folders / 72 .joblib artifacts


## 7. Optional: download a ZIP for Codex/Render

Models remain in Google Drive even if you skip this cell.

In [8]:
from google.colab import files
import shutil
zip_name = shutil.make_archive("/content/XGB_Mandalay_CPCB_Delta24_2026", "zip", root_dir=OUTPUT_DIR.parent, base_dir=OUTPUT_DIR.name)
print("ZIP:",zip_name)
files.download(zip_name)

ZIP: /content/XGB_Mandalay_CPCB_Delta24_2026.zip


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [9]:
from google.colab import files

files.download(
    "/content/XGB_Mandalay_CPCB_Delta24_2026.zip"
)

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [10]:
from pathlib import Path
import zipfile

zip_path = "/content/XGB_Mandalay_CPCB_Delta24_2026.zip"

with zipfile.ZipFile(zip_path) as z:
    names = z.namelist()

model_files = [
    name for name in names
    if name.endswith("xgboost_delta_model.joblib")
]

print("Trained XGBoost models:", len(model_files))
print("Total files:", len(names))
print("ZIP size:", round(
    Path(zip_path).stat().st_size / 1024**2, 2
), "MB")

Trained XGBoost models: 24
Total files: 99
ZIP size: 4.86 MB
